# MuseTalk v1.5 on Kaggle with Python 3.10

Kaggle's current notebook runtime is Python 3.12, but MuseTalk's official setup is based on Python 3.10 and PyTorch 2.0.1.

This notebook avoids that mismatch by creating a separate Python 3.10 environment under `/kaggle/working` with `uv`, so it does not rely on Conda being preinstalled in the Kaggle image.

Before running:
- set `Accelerator` to `GPU`
- set `Internet` to `ON`
- add a Kaggle dataset that contains your face image or face video and your audio file

Supported flow:
- `image + audio -> talking video`
- `video + audio -> dubbed video`

If you only have text, generate speech audio first with any TTS model, then feed that audio into MuseTalk.

In [ ]:
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_DIR = WORKDIR / 'MuseTalk'
RESULTS_DIR = WORKDIR / 'results'
ENV_PATH = WORKDIR / 'musetalk-py310'
ENV_PYTHON = ENV_PATH / 'bin' / 'python'

print('Workdir:', WORKDIR)
print('Repo dir:', REPO_DIR)
print('Env path:', ENV_PATH)

In [ ]:
%%bash
set -e
apt-get update -qq
apt-get install -y -qq ffmpeg git git-lfs

if [ ! -d /kaggle/working/MuseTalk ]; then
  git clone https://github.com/TMElyralab/MuseTalk.git /kaggle/working/MuseTalk
else
  echo "MuseTalk repo already exists, skipping clone."
fi

ffmpeg -version | head -n 1

In [ ]:
%%bash
set -e

apt-get update -qq
apt-get install -y -qq curl

ENV_PATH=/kaggle/working/musetalk-py310
ENV_PYTHON=$ENV_PATH/bin/python

if ! command -v uv >/dev/null 2>&1; then
  curl -LsSf https://astral.sh/uv/install.sh | sh
  hash -r
else
  echo "uv already installed, skipping bootstrap."
fi

UV_BIN=$(command -v uv || true)
if [ -z "$UV_BIN" ] && [ -x /usr/local/bin/uv ]; then
  UV_BIN=/usr/local/bin/uv
fi

if [ -z "$UV_BIN" ]; then
  echo "uv executable not found after installation."
  exit 1
else
  echo "Using uv at $UV_BIN"
fi

if [ ! -x "$ENV_PYTHON" ]; then
  "$UV_BIN" venv --python 3.10 --seed "$ENV_PATH"
else
  echo "Python 3.10 environment already exists, skipping creation."
fi

"$ENV_PYTHON" -V
"$UV_BIN" pip install --python "$ENV_PYTHON" -q --upgrade pip wheel "setuptools<82" tomli json-tricks
"$UV_BIN" pip install --python "$ENV_PYTHON" -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
"$UV_BIN" pip install --python "$ENV_PYTHON" -q --force-reinstall "numpy==1.23.5"
"$UV_BIN" pip install --python "$ENV_PYTHON" -q -r /kaggle/working/MuseTalk/requirements.txt
"$UV_BIN" pip install --python "$ENV_PYTHON" -q --no-cache-dir -U openmim gdown requests
"$UV_BIN" pip install --python "$ENV_PYTHON" -q "huggingface_hub==0.30.2"
"$UV_BIN" pip install --python "$ENV_PYTHON" -q "setuptools<82" "numpy==1.23.5" tomli json-tricks
"$ENV_PYTHON" -m mim install mmengine
"$ENV_PYTHON" -m mim install "mmcv==2.0.1"
"$ENV_PYTHON" -m mim install "mmdet==3.1.0"
"$UV_BIN" pip uninstall --python "$ENV_PYTHON" mmpose chumpy || true
"$UV_BIN" pip install --python "$ENV_PYTHON" -q --no-deps "mmpose==1.1.0"
"$UV_BIN" pip install --python "$ENV_PYTHON" -q cython xtcocotools munkres
"$UV_BIN" pip install --python "$ENV_PYTHON" -q --force-reinstall "numpy==1.23.5" "opencv-python==4.9.0.80"

In [ ]:
import subprocess

check_code = r'''
import torch
import mmcv
import mmdet
import mmpose
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print('MMCV:', mmcv.__version__)
print('MMDet:', mmdet.__version__)
print('MMPose:', mmpose.__version__)
'''

subprocess.run([str(ENV_PYTHON), '-c', check_code], check=True)

In [ ]:
import subprocess

download_code = r'''
from pathlib import Path
from huggingface_hub import snapshot_download
import requests
import gdown

repo_dir = Path('/kaggle/working/MuseTalk')
models_dir = repo_dir / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

for subdir in [
    'musetalk',
    'musetalkV15',
    'syncnet',
    'dwpose',
    'face-parse-bisent',
    'sd-vae',
    'whisper',
]:
    (models_dir / subdir).mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id='TMElyralab/MuseTalk',
    local_dir=str(models_dir),
    allow_patterns=[
        'musetalk/musetalk.json',
        'musetalk/pytorch_model.bin',
        'musetalkV15/musetalk.json',
        'musetalkV15/unet.pth',
    ],
    local_dir_use_symlinks=False,
)

snapshot_download(
    repo_id='stabilityai/sd-vae-ft-mse',
    local_dir=str(models_dir / 'sd-vae'),
    allow_patterns=['config.json', 'diffusion_pytorch_model.bin'],
    local_dir_use_symlinks=False,
)

snapshot_download(
    repo_id='openai/whisper-tiny',
    local_dir=str(models_dir / 'whisper'),
    allow_patterns=['config.json', 'pytorch_model.bin', 'preprocessor_config.json'],
    local_dir_use_symlinks=False,
)

snapshot_download(
    repo_id='yzd-v/DWPose',
    local_dir=str(models_dir / 'dwpose'),
    allow_patterns=['dw-ll_ucoco_384.pth'],
    local_dir_use_symlinks=False,
)

snapshot_download(
    repo_id='ByteDance/LatentSync',
    local_dir=str(models_dir / 'syncnet'),
    allow_patterns=['latentsync_syncnet.pt'],
    local_dir_use_symlinks=False,
)

face_parse_path = models_dir / 'face-parse-bisent' / '79999_iter.pth'
if not face_parse_path.exists():
    gdown.download(
        id='154JgKpzCPW82qINcVieuPH3fZ2e0P812',
        output=str(face_parse_path),
        quiet=False,
    )

resnet_path = models_dir / 'face-parse-bisent' / 'resnet18-5c106cde.pth'
if not resnet_path.exists():
    with requests.get('https://download.pytorch.org/models/resnet18-5c106cde.pth', stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(resnet_path, 'wb') as handle:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)

print('Weights downloaded to:', models_dir)
'''

subprocess.run([str(ENV_PYTHON), '-c', download_code], check=True)

## Set your input paths

Point `VIDEO_OR_IMAGE_PATH` and `AUDIO_PATH` to files in `/kaggle/input/...`.

Examples:
- `/kaggle/input/my-data/person.mp4`
- `/kaggle/input/my-data/person.jpg`
- `/kaggle/input/my-data/voice.wav`

For the first run, use a short clip or a single portrait image plus a short audio file.

In [ ]:
from pathlib import Path

MODEL_VERSION = 'v15'  # 'v15' or 'v1'
VIDEO_OR_IMAGE_PATH = '/kaggle/input/your-dataset-name/input_video.mp4'
AUDIO_PATH = '/kaggle/input/your-dataset-name/input_audio.wav'
OUTPUT_NAME = 'musetalk_output.mp4'
FPS = 25
BATCH_SIZE = 4
USE_FLOAT16 = True
BBOX_SHIFT = -7  # only meaningful for v1

assert Path(VIDEO_OR_IMAGE_PATH).exists(), f'Input face file not found: {VIDEO_OR_IMAGE_PATH}'
assert Path(AUDIO_PATH).exists(), f'Audio file not found: {AUDIO_PATH}'

print('Face input:', VIDEO_OR_IMAGE_PATH)
print('Audio input:', AUDIO_PATH)

In [ ]:
config_path = REPO_DIR / 'configs' / 'inference' / 'kaggle_test.yaml'

lines = [
    'task_0:',
    f'  video_path: "{VIDEO_OR_IMAGE_PATH}"',
    f'  audio_path: "{AUDIO_PATH}"',
    f'  result_name: "{OUTPUT_NAME}"',
]

if MODEL_VERSION == 'v1':
    lines.append(f'  bbox_shift: {BBOX_SHIFT}')

config_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(config_path.read_text(encoding='utf-8'))

In [ ]:
import os
import shlex
import subprocess

unet_model_path = REPO_DIR / ('models/musetalkV15/unet.pth' if MODEL_VERSION == 'v15' else 'models/musetalk/pytorch_model.bin')
unet_config = REPO_DIR / ('models/musetalkV15/musetalk.json' if MODEL_VERSION == 'v15' else 'models/musetalk/musetalk.json')

command = [
    str(ENV_PYTHON),
    '-m', 'scripts.inference',
    '--inference_config', str(config_path),
    '--result_dir', str(RESULTS_DIR),
    '--unet_model_path', str(unet_model_path),
    '--unet_config', str(unet_config),
    '--version', MODEL_VERSION,
    '--fps', str(FPS),
    '--batch_size', str(BATCH_SIZE),
    '--ffmpeg_path', '/usr/bin',
]

if USE_FLOAT16:
    command.append('--use_float16')

print('Running command:')
print(' '.join(shlex.quote(part) for part in command))

env = os.environ.copy()
env['MPLBACKEND'] = 'Agg'

subprocess.run(command, cwd=str(REPO_DIR), env=env, check=True)

In [ ]:
from IPython.display import Video, display

output_files = sorted(RESULTS_DIR.rglob('*.mp4'), key=lambda path: path.stat().st_mtime)
assert output_files, f'No mp4 output found under {RESULTS_DIR}'

final_output = output_files[-1]
print('Output video:', final_output)
display(Video(str(final_output), embed=True))